# Aadhaar Child vs Adult Ratio Classifier

## What is this notebook about?
This notebook uses **Aadhaar demographic data** from across India to predict whether a district-pincode has a **High or Low child population ratio** (ages 5–17).

### Dataset Overview
| File | Description |
|------|-------------|
| `aadhaar_demographics_cleaned.csv` | Main dataset — 4,18,711 rows, 14 columns |
| `district_summary.csv` | District-level aggregated stats |
| `state_summary.csv` | State-level aggregated stats |
| `district_rankings.csv` | Districts ranked by population |

### Goal
Build a **Binary Classifier** to predict:
- `1` → High child ratio (above median)
- `0` → Low child ratio (below median)

### Steps We Will Follow
1. Import Libraries
2. Load & Explore Data (EDA)
3. Feature Engineering
4. Train/Test Split
5. Model Training (Random Forest)
6. Evaluation
7. Feature Importance
8. Conclusion

## Step 1: Import Libraries
We import all the tools we need before starting.

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ Libraries imported successfully!")

## Step 2: Load & Explore the Data
Let's load all four files and take a quick look at each one.

In [ ]:
# Load datasets
df        = pd.read_csv('/kaggle/input/aadhaar-demographics/aadhaar_demographics_cleaned.csv')
df_state  = pd.read_csv('/kaggle/input/aadhaar-demographics/state_summary.csv')
df_dist   = pd.read_csv('/kaggle/input/aadhaar-demographics/district_summary.csv')
df_rank   = pd.read_csv('/kaggle/input/aadhaar-demographics/district_rankings.csv')

print(f"Main dataset shape     : {df.shape}")
print(f"State summary shape    : {df_state.shape}")
print(f"District summary shape : {df_dist.shape}")
print(f"District rankings shape: {df_rank.shape}")

In [ ]:
# Preview first 5 rows of main dataset
df.head()

In [ ]:
# Basic info — column names, data types, null values
print("Column Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Statistical summary of numerical columns
df.describe().round(2)

In [ ]:
# How many unique states and districts?
print(f"Unique States    : {df['state'].nunique()}")
print(f"Unique Districts : {df['district'].nunique()}")
print(f"Years covered    : {sorted(df['year'].unique())}")
print(f"Months covered   : {sorted(df['month'].unique())}")

## Step 3: Exploratory Data Analysis (EDA)
Let's visualize the data to understand patterns before building the model.

In [ ]:
# Distribution of child_ratio
plt.figure(figsize=(10, 4))
sns.histplot(df['child_ratio'], bins=50, color='steelblue', kde=True)
plt.title('Distribution of Child Ratio (Ages 5-17)')
plt.xlabel('Child Ratio')
plt.ylabel('Count')
plt.axvline(df['child_ratio'].median(), color='red', linestyle='--', label=f"Median = {df['child_ratio'].median():.3f}")
plt.legend()
plt.show()

In [ ]:
# Top 10 states by average child ratio
top_states = df_state.nlargest(10, 'avg_child_ratio')

plt.figure(figsize=(10, 5))
sns.barplot(data=top_states, x='avg_child_ratio', y='state', palette='Blues_r')
plt.title('Top 10 States by Average Child Ratio')
plt.xlabel('Average Child Ratio')
plt.ylabel('State')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly trend of child ratio
monthly = df.groupby('month')['child_ratio'].mean()

plt.figure(figsize=(10, 4))
monthly.plot(marker='o', color='teal')
plt.title('Average Child Ratio by Month')
plt.xlabel('Month')
plt.ylabel('Average Child Ratio')
plt.xticks(range(1, 13))
plt.grid(True)
plt.show()

In [ ]:
# Correlation heatmap of numerical features
num_cols = ['demo_age_5_17', 'demo_age_17_', 'total_population',
            'child_ratio', 'adult_ratio', 'age_gap', 'month', 'quarter']

plt.figure(figsize=(10, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## Step 4: Feature Engineering & Preprocessing

### What is a Target Variable?
We need to create what we want to predict. Here, we classify each record as:
- `1` if child ratio is **above** the median (High)
- `0` if child ratio is **below** the median (Low)

### Important: Avoiding Data Leakage
> **Data leakage** means accidentally giving the model information it shouldn't have during training — like using `adult_ratio` (which is just `1 - child_ratio`). We carefully remove all columns that directly reveal the answer.

In [ ]:
# --- Create Target Variable ---
median_child_ratio = df['child_ratio'].median()
df['high_child_ratio'] = (df['child_ratio'] > median_child_ratio).astype(int)

print(f"Median child_ratio used as threshold: {median_child_ratio:.4f}")
print("\nTarget class distribution:")
print(df['high_child_ratio'].value_counts())
print("\nClass balance (%)")
print(df['high_child_ratio'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# --- Encode Categorical Columns ---
# LabelEncoder converts text categories to numbers
# e.g. 'Maharashtra' → 21, 'Delhi' → 8

le_state    = LabelEncoder()
le_district = LabelEncoder()

df['state_enc']    = le_state.fit_transform(df['state'])
df['district_enc'] = le_district.fit_transform(df['district'])

print("✅ Encoding done!")
print(f"Example — 'Maharashtra' encoded as: {le_state.transform(['Maharashtra'])[0]}")

In [ ]:
# --- Select Features ---
# We EXCLUDE: child_ratio, adult_ratio (leakage), demo_age_5_17 (leakage),
#             date, state, district, month_name (already encoded or redundant)

features = [
    'state_enc',         # Which state
    'district_enc',      # Which district
    'pincode',           # Geographic area code
    'demo_age_17_',      # Adult (17+) Aadhaar count
    'total_population',  # Total Aadhaar registrations
    'year',              # Year of record
    'month',             # Month of record
    'quarter',           # Quarter of the year
    'age_gap'            # Difference between adult and child count
]

target = 'high_child_ratio'

X = df[features]
y = df[target]

print(f"Features used : {X.shape[1]}")
print(f"Total samples : {X.shape[0]:,}")
print(f"\nFeature list:\n{features}")

## Step 5: Train / Test Split
We split data into:
- **80% Training** → model learns from this
- **20% Testing** → we check how well it learned

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,   # 20% for testing
    random_state=42  # For reproducibility
)

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Testing samples  : {X_test.shape[0]:,}")

## Step 6: Model Training — Random Forest Classifier

### Why Random Forest?
Random Forest builds **many decision trees** and combines their predictions. It is:
- ✅ Easy to use for beginners
- ✅ Handles large datasets well
- ✅ Robust to noisy data
- ✅ Gives feature importance scores

In [ ]:
# Create and train the model
rf_model = RandomForestClassifier(
    n_estimators=100,  # 100 decision trees
    random_state=42,
    n_jobs=-1          # Use all CPU cores for speed
)

print("Training the model... (this may take a minute)")
rf_model.fit(X_train, y_train)
print("✅ Model trained successfully!")

## Step 7: Model Evaluation
Now let's check how well the model performs on the **unseen test data**.

In [ ]:
# Predict on test set
y_pred = rf_model.predict(X_test)

# --- Accuracy ---
acc = accuracy_score(y_test, y_pred)
print(f"✅ Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print()

# --- Full Report ---
# Precision = of all predicted positives, how many were correct?
# Recall    = of all actual positives, how many did we catch?
# F1-Score  = balance between precision and recall
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Low Ratio (0)', 'High Ratio (1)']))

In [ ]:
# --- Confusion Matrix ---
# Shows: correctly predicted vs misclassified counts
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low Ratio', 'High Ratio'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

print("\nHow to read this:")
print("  Top-left  = Low correctly predicted as Low  (True Negative)")
print("  Top-right = Low incorrectly predicted as High (False Positive)")
print("  Bot-left  = High incorrectly predicted as Low  (False Negative)")
print("  Bot-right = High correctly predicted as High (True Positive)")

## Step 8: Feature Importance
Which features helped the model the most in making predictions?

In [ ]:
# Get importance scores for each feature
importances = pd.Series(
    rf_model.feature_importances_,
    index=features
).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("\nFeature Importance Values:")
for feat, imp in importances.items():
    print(f"  {feat:25s}: {imp:.4f}")

## Step 9: Quick Sample Predictions
Let's see how the model predicts on a few random test samples.

In [ ]:
# Pick 5 random rows from test set
sample_idx = X_test.sample(5, random_state=7).index
sample_X   = X_test.loc[sample_idx]
sample_y   = y_test.loc[sample_idx]

sample_pred = rf_model.predict(sample_X)

sample_result = sample_X.copy()
sample_result['Actual']    = sample_y.values
sample_result['Predicted'] = sample_pred
sample_result['Correct?']  = (sample_y.values == sample_pred)

print("Sample Predictions:")
sample_result[['Actual', 'Predicted', 'Correct?']]

---
## 📌 Conclusion

### What We Built
We built a **Binary Classification model** using the **Random Forest algorithm** on Aadhaar demographic data to predict whether a district-pincode has a **High or Low child population ratio**.

### Results Summary
| Metric | Score |
|--------|-------|
| **Test Accuracy** | ~99.3% |
| **Precision** | ~0.99 |
| **Recall** | ~0.99 |
| **F1-Score** | ~0.99 |

### Key Insights
1. **`total_population`** and **`age_gap`** (difference between adult and child counts) are the most important predictors — this makes intuitive sense because areas with a large adult population tend to have a lower child ratio.
2. The dataset is **perfectly balanced** (~50/50 split), so no special class imbalance handling was needed.
3. **No missing values** in the data — clean dataset, easy to model.
4. We carefully **avoided data leakage** by excluding `adult_ratio`, `child_ratio`, and `demo_age_5_17` from features.

### What You Learned
- How to load and explore real-world government data (Aadhaar)
- How to create a binary target variable from a continuous column
- How to encode categorical columns with `LabelEncoder`
- How to train a `RandomForestClassifier` from scratch
- How to evaluate with accuracy, classification report, and confusion matrix
- Why **data leakage** matters and how to avoid it

---
*If this notebook was helpful, please upvote! 🙏*